# SHAP 값 실습

**SHAP · Shapley Value · 섀플리 값**

예측값을 각 입력 변수의 기여로 나누어 설명하는 값. 개별 예측 단위의 해석에 쓰인다.

소재 분야에서 이해하기: 한 조성의 예측 강도가 높게 나온 이유를 변수별 기여로 본다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [SHAP 문서](https://shap.readthedocs.io/en/latest/)

## 1. 예측 하나를 변수 기여로 쪼개기

shap 패키지 없이, 선형 모델에서 정확한 섀플리 값을 직접 계산합니다.
선형 모델의 섀플리 값은 계수 × (값 - 평균) 입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(X, y)
baseline = model.predict(X.mean(0, keepdims=True))[0]
sample = X[0:1]
contributions = model.coef_ * (sample[0] - X.mean(0))
print('기준값(평균 입력의 예측) %.2f HV' % baseline)
for name, value in zip(FEATURES, contributions):
    print('  %-8s 기여 %+.2f HV' % (name, value))
print('기여 합 + 기준값 = %.2f, 실제 예측 %.2f' % (baseline + contributions.sum(), model.predict(sample)[0]))

## 2. 비선형 모델은 표본으로 근사합니다

변수 순서를 무작위로 섞어가며 기여를 평균하는 방식(순열 섀플리)을 구현합니다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import itertools

forest = RandomForestRegressor(n_estimators=200, random_state=0).fit(X, y)
background = X[rng.choice(len(X), 80, replace=False)]

def shapley(target, permutations=120, seed=0):
    local = np.random.default_rng(seed)
    total = np.zeros(X.shape[1])
    for _ in range(permutations):
        order = local.permutation(X.shape[1])
        current = background.copy()
        previous = forest.predict(current).mean()
        for index in order:
            current[:, index] = target[index]
            value = forest.predict(current).mean()
            total[index] += value - previous
            previous = value
    return total / permutations

values = shapley(X[0])
plt.barh(range(len(FEATURES)), values)
plt.yticks(range(len(FEATURES)), ['temp', 'time', 'additive', 'noise'])
plt.xlabel('SHAP value (HV)'); plt.axvline(0, color='k', lw=1); plt.show()
print('기준값 %.2f + 기여 합 %.2f = %.2f (실제 예측 %.2f)'
      % (forest.predict(background).mean(), values.sum(),
         forest.predict(background).mean() + values.sum(), forest.predict(X[0:1])[0]))

## 3. 해석

섀플리 값은 기준값에서 이 예측까지의 차이를 변수들에게 공정하게 나눠줍니다.
합이 예측과 기준값의 차이와 같아지는 성질(가법성) 때문에 개별 예측 설명에 쓰기 좋습니다.
다만 계산 비용이 크고, 상관된 변수에서는 해석에 주의가 필요합니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#shap)을 여세요.